<a href="https://colab.research.google.com/github/pachterlab/cellmender/blob/main/runtime.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# try:
#     from cellmender import denoise_count_matrix
# except ImportError:
#     print("cellmender not found, installing...")
#     !pip install -U -q cellmender[analysis]

In [5]:
import os
import time
import subprocess
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from cellmender import denoise_count_matrix
import cellmender.utils as cm_utils

cellmender_dir = os.path.dirname(os.path.abspath(""))
rver_docker_workspace = "/home/ruser/work/cellmender"

# Runtime

pbmc8k dataset: 8k PBMCs from a healthy donor (CellBender Fig2): https://www.10xgenomics.com/datasets/8-k-pbm-cs-from-a-healthy-donor-2-standard-2-1-0

In [6]:
dataset_name = "pbmc8k"  # options: pbmc8k, hgmm12k, tiny_cellbender, simulation1, custom
verbose = 2  # 2 debug, 1 info, 0 warning, -1 error, -2 critical
scar_env = "/home/jrich/miniconda3/envs/scar"

In [7]:
data_dir = os.path.join(cellmender_dir, "notebooks", "data", dataset_name, "runtime")
os.makedirs(data_dir, exist_ok=True)

out_dir = os.path.join(cellmender_dir, "notebooks", "output", dataset_name, "runtime")
os.makedirs(out_dir, exist_ok=True)

if dataset_name == "pbmc8k":
    adata_path_raw = f"{data_dir}/pbmc8k_raw_gene_bc_matrices_h5.h5"
    sequencing_technology = "10XV2"
    model_pkl = "Immune_All_High.pkl"  # path to celltypist model pkl file
    expected_cells = 8381

    if not os.path.exists(adata_path_raw):
        !wget -O {adata_path_raw} https://cf.10xgenomics.com/samples/cell-exp/2.1.0/pbmc8k/pbmc8k_raw_gene_bc_matrices_h5.h5

    matrix_tar_files_dir = os.path.join(data_dir, "matrix_tar_files")
    os.makedirs(matrix_tar_files_dir, exist_ok=True)
    raw_tar_file_dir = os.path.join(matrix_tar_files_dir, "raw_gene_bc_matrices", "GRCh38")
    filtered_tar_file_dir = os.path.join(matrix_tar_files_dir, "filtered_gene_bc_matrices", "GRCh38")
    if not os.path.exists(raw_tar_file_dir):
        raw_tar_path = os.path.join(matrix_tar_files_dir, "pbmc8k_raw_gene_bc_matrices.tar.gz")
        !wget -O {raw_tar_path} https://cf.10xgenomics.com/samples/cell-exp/2.1.0/pbmc8k/pbmc8k_raw_gene_bc_matrices.tar.gz
        !tar -xvzf {raw_tar_path} -C {matrix_tar_files_dir}
    if not os.path.exists(filtered_tar_file_dir):
        filtered_tar_path = os.path.join(matrix_tar_files_dir, "pbmc8k_filtered_gene_bc_matrices.tar.gz")
        !wget -O {filtered_tar_path} https://cf.10xgenomics.com/samples/cell-exp/2.1.0/pbmc8k/pbmc8k_filtered_gene_bc_matrices.tar.gz
        !tar -xvzf {filtered_tar_path} -C {matrix_tar_files_dir}
else:
    raise ValueError(f"Dataset name {dataset_name} not recognized.")

min_genes = 0
min_cells = 0
umi_top_percentile_to_remove = 5
unique_genes_top_percentile_to_remove = 5
mt_gene_percentile_to_remove = 10
max_mt_percentage = None
n_top_genes = 2000
n_pcs = 25
n_neighbors = 20
leiden_resolution = 1.0

has_gpu = subprocess.call("nvidia-smi", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) == 0
max_threads = os.cpu_count()

tool_to_runtime_dict = {}

--2025-11-22 00:42:31--  https://cf.10xgenomics.com/samples/cell-exp/2.1.0/pbmc8k/pbmc8k_raw_gene_bc_matrices_h5.h5
Resolving cf.10xgenomics.com (cf.10xgenomics.com)... 104.18.1.173, 104.18.0.173, 2606:4700::6812:ad, ...
Connecting to cf.10xgenomics.com (cf.10xgenomics.com)|104.18.1.173|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 32549555 (31M) [binary/octet-stream]
Saving to: ‘/data/cellmender/notebooks/data/pbmc8k/runtime/pbmc8k_raw_gene_bc_matrices_h5.h5’

/data/cellmender/no 100%[===================>]  31.04M  22.3MB/s    in 1.4s    

2025-11-22 00:42:33 (22.3 MB/s) - ‘/data/cellmender/notebooks/data/pbmc8k/runtime/pbmc8k_raw_gene_bc_matrices_h5.h5’ saved [32549555/32549555]

--2025-11-22 00:42:33--  https://cf.10xgenomics.com/samples/cell-exp/2.1.0/pbmc8k/pbmc8k_raw_gene_bc_matrices.tar.gz
Resolving cf.10xgenomics.com (cf.10xgenomics.com)... 104.18.0.173, 104.18.1.173, 2606:4700::6812:1ad, ...
Connecting to cf.10xgenomics.com (cf.10xgenomics.com)|104.

## Raw

In [8]:
adata_raw = cm_utils.load_adata(adata_path_raw, verbose=verbose)
adata_raw.var_names_make_unique()

adata_raw = cm_utils.infer_empty_droplets(adata_raw, method="threshold", expected_cells=expected_cells, verbose=verbose)  # adds adata.obs["is_empty"]

if "celltype" not in adata_raw.obs.columns:
    adata_raw = cm_utils.determine_cell_types(adata_raw, model_pkl=model_pkl, filter_empty=True, expected_cells=expected_cells, verbose=verbose)

00:42:37 - INFO - Loading adata from '/data/cellmender/notebooks/data/pbmc8k/runtime/pbmc8k_raw_gene_bc_matrices_h5.h5'
/home/jrich/miniconda3/envs/cellmender/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
00:42:38 - INFO - Filtering empty droplets using column 'is_empty' in adata.obs. If this column is not present, it will be inferred using method 'celltypist' with umi_cutoff=None and expected_cells=8381.
00:42:38 - INFO - Running cell type annotation using CellTypist with model_pkl=Immune_All_High.pkl. This may take some time depending on the size of the dataset and the model used.
/home/jrich/miniconda3/envs/cellmender/lib/python3.10/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv
📂 Storing models in /home/jrich

## cellmender

In [9]:
threads = 1

if threads > max_threads:
    SystemExit(f"Requested {threads} threads but only {max_threads} available.")

cellmender_max_iter = 150
adata_path_cellmender = os.path.join(data_dir, f"pbmc8k_cellmender_{threads}threads.h5ad")
cellmender_log_file = os.path.join(data_dir, f"pbmc8k_cellmender_{threads}threads.log")

adata = adata_raw.copy()
if "celltype" not in adata.obs.columns:
    adata = cm_utils.determine_cell_types(adata, model_pkl=model_pkl, filter_empty=True, expected_cells=expected_cells, verbose=verbose)
start_time = time.perf_counter()
_ = denoise_count_matrix(adata, adata_out=adata_path_cellmender, max_iter=cellmender_max_iter, beta=0.03, eps=1e-9, empty_droplet_method="threshold", expected_cells=expected_cells, cell_ambient_fraction=0.01, threads=threads, verbose=verbose, log_file=cellmender_log_file)
cellmender_runtime = time.perf_counter() - start_time
tool_to_runtime_dict[f"cellmender_{threads}thread"] = cellmender_runtime

cm_utils.plot_cellmender_likelihood_over_epochs(log_path=cellmender_log_file, show=True)

00:43:30 - INFO - Inferring gene ambient fractions.
00:43:30 - INFO - Added 'ambient_fraction' to adata.var.
00:43:30 - INFO - Inferring celltype profiles.


Logging to /data/cellmender/notebooks/data/pbmc8k/runtime/pbmc8k_cellmender_1threads.log


00:43:30 - INFO - Number of parameters in the cellmender model: 344,982 (alpha_i: 8,381, beta: 1, gamma_type: 67,048, p_k: 269,552)
00:43:33 - INFO - adata.obs does not have 'cell_ambient_fraction'. Setting to `cell_ambient_fraction` argument.
00:43:33 - INFO - Performing Sparse EM with 1 Numba thread(s)


KeyboardInterrupt: 

In [ ]:
threads = 16

if threads > max_threads:
    SystemExit(f"Requested {threads} threads but only {max_threads} available.")
    
cellmender_max_iter = 150
adata_path_cellmender = os.path.join(data_dir, f"cellmender_{threads}threads.h5ad")
cellmender_log_file = os.path.join(data_dir, f"cellmender_{threads}threads.log")

adata = adata_raw.copy()
if "celltype" not in adata.obs.columns:
    adata = cm_utils.determine_cell_types(adata, model_pkl=model_pkl, filter_empty=True, expected_cells=expected_cells, verbose=verbose)
start_time = time.perf_counter()
_ = denoise_count_matrix(adata, adata_out=adata_path_cellmender, max_iter=cellmender_max_iter, beta=0.03, eps=1e-9, empty_droplet_method="threshold", expected_cells=expected_cells, cell_ambient_fraction=0.01, threads=threads, verbose=verbose, log_file=cellmender_log_file)
cellmender_runtime = time.perf_counter() - start_time
tool_to_runtime_dict[f"cellmender_{threads}thread"] = cellmender_runtime

cm_utils.plot_cellmender_likelihood_over_epochs(log_path=cellmender_log_file, show=True)

## CellBender (v0.3.0)

In [ ]:
use_cuda = False
threads = 1


if use_cuda and not has_gpu:
     SystemExit("CUDA requested but no GPU available.")
if not use_cuda and threads > max_threads:
    SystemExit(f"Requested {threads} threads but only {max_threads} available.")

epochs = 150
runtime = "--cuda" if use_cuda else f"--cpu-threads {threads}"
gpus = "--gpus all" if use_cuda else ""
input_path = adata_path_raw.replace(f"{cellmender_dir}/notebooks/data", "/data")
output_path = os.path.join(data_dir, f"cellbender_gpu.h5").replace(f"{cellmender_dir}/notebooks/data", "/data") if use_cuda else os.path.join(data_dir, f"cellbender_cpu_{threads}threads.h5").replace(f"{cellmender_dir}/notebooks/data", "/data")

start_time = time.perf_counter()
!docker run --rm {gpus} -v {cellmender_dir}/notebooks/data:/data us.gcr.io/broad-dsde-methods/cellbender:0.3.0 \
     cellbender remove-background \
     --input {input_path} \
     --output {output_path} \
     --epochs {epochs} \
     --fpr 0.01 \
     --model full \
     {runtime}
cellbender_runtime = time.perf_counter() - start_time
dict_key = f"cellbender_{'gpu' if use_cuda else f'{threads}thread'}"
tool_to_runtime_dict[dict_key] = cellbender_runtime

In [ ]:
use_cuda = False
threads = 16

if use_cuda and not has_gpu:
     SystemExit("CUDA requested but no GPU available.")
if not use_cuda and threads > max_threads:
    SystemExit(f"Requested {threads} threads but only {max_threads} available.")

epochs = 150
runtime = "--cuda" if use_cuda else f"--cpu-threads {threads}"
gpus = "--gpus all" if use_cuda else ""
input_path = adata_path_raw.replace(f"{cellmender_dir}/notebooks/data", "/data")
output_path = os.path.join(data_dir, f"cellbender_gpu.h5").replace(f"{cellmender_dir}/notebooks/data", "/data") if use_cuda else os.path.join(data_dir, f"cellbender_cpu_{threads}threads.h5").replace(f"{cellmender_dir}/notebooks/data", "/data")

start_time = time.perf_counter()
!docker run --rm {gpus} -v {cellmender_dir}/notebooks/data:/data us.gcr.io/broad-dsde-methods/cellbender:0.3.0 \
     cellbender remove-background \
     --input {input_path} \
     --output {output_path} \
     --epochs {epochs} \
     --fpr 0.01 \
     --model full \
     {runtime}
cellbender_runtime = time.perf_counter() - start_time
dict_key = f"cellbender_{'gpu' if use_cuda else f'{threads}thread'}"
tool_to_runtime_dict[dict_key] = cellbender_runtime

In [ ]:
use_cuda = True
threads = None

if use_cuda and not has_gpu:
     SystemExit("CUDA requested but no GPU available.")
if not use_cuda and threads > max_threads:
    SystemExit(f"Requested {threads} threads but only {max_threads} available.")

epochs = 150
runtime = "--cuda" if use_cuda else f"--cpu-threads {threads}"
gpus = "--gpus all" if use_cuda else ""
input_path = adata_path_raw.replace(f"{cellmender_dir}/notebooks/data", "/data")
output_path = os.path.join(data_dir, f"cellbender_gpu.h5").replace(f"{cellmender_dir}/notebooks/data", "/data") if use_cuda else os.path.join(data_dir, f"cellbender_cpu_{threads}threads.h5").replace(f"{cellmender_dir}/notebooks/data", "/data")

start_time = time.perf_counter()
!docker run --rm {gpus} -v {cellmender_dir}/notebooks/data:/data us.gcr.io/broad-dsde-methods/cellbender:0.3.0 \
     cellbender remove-background \
     --input {input_path} \
     --output {output_path} \
     --epochs {epochs} \
     --fpr 0.01 \
     --model full \
     {runtime}
cellbender_runtime = time.perf_counter() - start_time
dict_key = f"cellbender_{'gpu' if use_cuda else f'{threads}thread'}"
tool_to_runtime_dict[dict_key] = cellbender_runtime

## SoupX (v1.6.2)

In [ ]:
adata_soupx_obs_csv = f"{data_dir}/soupx_obs.csv"
if not os.path.exists(adata_soupx_obs_csv):
    adata_soupx_tmp = cm_utils.load_adata(filtered_tar_file_dir)
    adata_soupx_tmp = cm_utils.run_scanpy_preprocessing_and_clustering(adata_soupx_tmp, min_genes=min_genes, min_cells=min_cells, max_mt_percentage=max_mt_percentage, n_top_genes=n_top_genes, n_pcs=n_pcs, n_neighbors=n_neighbors, leiden_resolution=leiden_resolution, seed=42, verbose=verbose)
    adata_soupx_tmp.obs[["leiden"]].to_csv(adata_soupx_obs_csv)

soupx_out_prefix = os.path.join(data_dir, f"soupX")

start_time = time.perf_counter()
!docker run --rm \
    -w /home/ruser/work \
    -v {cellmender_dir}:{rver_docker_workspace} \
    josephrich98/cellmender_tutorials:soupx.0.1.0 \
    Rscript {rver_docker_workspace}/scripts/run_soupx.R \
        {matrix_tar_files_dir_soupx.replace(cellmender_dir, rver_docker_workspace)} \
        {adata_soupx_obs_csv.replace(cellmender_dir, rver_docker_workspace)} \
        {soupx_out_prefix.replace(cellmender_dir, rver_docker_workspace)} \
        leiden
soupx_runtime = time.perf_counter() - start_time
tool_to_runtime_dict["soupx"] = soupx_runtime

## DecontX (v1.8.0)

In [ ]:
decontx_out_prefix = os.path.join(data_dir, f"decontX")

start_time = time.perf_counter()
!docker run --rm \
    -w /home/ruser/work \
    -v {cellmender_dir}:{rver_docker_workspace} \
    josephrich98/cellmender_tutorials:decontx.0.1.0 \
    Rscript {rver_docker_workspace}/scripts/run_decontx.R \
        {raw_tar_file_dir.replace(cellmender_dir, rver_docker_workspace)} \
        {filtered_tar_file_dir.replace(cellmender_dir, rver_docker_workspace)} \
        {sequencing_technology} \
        {decontx_out_prefix.replace(cellmender_dir, rver_docker_workspace)} \
        --dont_prepend_sample_to_barcodes
decontx_runtime = time.perf_counter() - start_time
tool_to_runtime_dict["decontx"] = decontx_runtime

## scAR (v0.7.0)

In [ ]:
%env MPLBACKEND=
use_cuda = False

if use_cuda and not has_gpu:
     SystemExit("CUDA requested but no GPU available.")

epochs = 150
adata_path_scar = os.path.join(data_dir, f"scar_gpu.h5ad") if use_cuda else os.path.join(data_dir, f"scar_cpu_{threads}threads.h5ad")

runtime = "--cuda" if use_cuda else ""
conda_run_flag = "-p" if "/" in scar_env else "-n"
start_time = time.perf_counter()
!conda run {conda_run_flag} {scar_env} \
     python {cellmender_dir}/scripts/run_scar.py \
     -r {raw_tar_file_dir} \
     -f {filtered_tar_file_dir} \
     -o {adata_path_scar} \
     {runtime} \
     --epochs {epochs}
scar_runtime = time.perf_counter() - start_time
dict_key = f"scar_{'gpu' if use_cuda else f'{threads}thread'}"
tool_to_runtime_dict[dict_key] = scar_runtime

In [ ]:
%env MPLBACKEND=
use_cuda = True

if use_cuda and not has_gpu:
     SystemExit("CUDA requested but no GPU available.")

epochs = 150
adata_path_scar = os.path.join(data_dir, f"scar_gpu.h5ad") if use_cuda else os.path.join(data_dir, f"scar_cpu_{threads}threads.h5ad")

runtime = "--cuda" if use_cuda else ""
conda_run_flag = "-p" if "/" in scar_env else "-n"
start_time = time.perf_counter()
!conda run {conda_run_flag} {scar_env} \
     python {cellmender_dir}/scripts/run_scar.py \
     -r {raw_tar_file_dir} \
     -f {filtered_tar_file_dir} \
     -o {adata_path_scar} \
     {runtime} \
     --epochs {epochs}
scar_runtime = time.perf_counter() - start_time
dict_key = f"scar_{'gpu' if use_cuda else f'{threads}thread'}"
tool_to_runtime_dict[dict_key] = scar_runtime

In [ ]:
# convert to minutes
tool_to_runtime_minutes_dict = {k: v / 60 for k, v in tool_to_runtime_dict.items()}
print(tool_to_runtime_minutes_dict)

In [ ]:
plt.figure(figsize=(8,6))
sns.barplot(x=list(tool_to_runtime_minutes_dict.keys()), y=list(tool_to_runtime_minutes_dict.values()), color="gray")
plt.ylabel("Runtime (minutes)")
plt.savefig(os.path.join(out_dir, "runtime_comparison.png"))
plt.show()